# pipeline

In [ ]:
import pandas as pd
import transformers
from contextlib import contextmanager
import torch
import gc
from transformers import pipeline
from datasets import Dataset
import numpy as np

# read dataset
def read_dataset(dataset, strategy):
    path = f'/homes/lst20/fyp/fyp_resources/LexEval-main/analysis/df_{strategy.replace('-', '')}_{dataset}_.csv'
    df = pd.read_csv(path)
    return df

def clear_cache():
    """ free CUDA & Python memory."""
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                del obj
        except Exception:  # pragma: no cover – best‑effort cleanup
            pass
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()  # Ensure all kernels are finished
    


In [ ]:
from contextlib import contextmanager
from datasets import Dataset
import torch
import ast

@contextmanager
def get_qnli():
    pipe = pipeline(
        "text-classification",
        model="textattack/roberta-base-QNLI",
        batch_size=512,
        return_all_scores=True
    )
    try:
        yield pipe
    finally:
        del pipe
        torch.cuda.empty_cache()

@contextmanager
def get_mnli():
    pipe = pipeline(
        "text-classification",
        model="facebook/bart-large-mnli",
        batch_size=512,
        return_all_scores=True
    )
    try:
        yield pipe
    finally:
        del pipe
        torch.cuda.empty_cache()


def evaluate_pipeline(df_list, declarative_generator):
    from spacy.lang.en import English
    import re, ast
    nlp = English()
    nlp.add_pipe("sentencizer")

    def extract_sentences(text):
        sentences = []
        for segment in text.split("\n"):
            doc = nlp(segment.strip())
            for sent in doc.sents:
                cleaned = sent.text.strip()
                if re.search(r'\w', cleaned):
                    sentences.append(cleaned)
        return sentences

    updated_dfs = []

    for original_df in df_list:
        # Step 0: compute root_prompt
        df = original_df.copy()
        df.loc[:, 'root_prompt'] = df.apply(
            lambda row: row['prompt']
                        if row['type'] == 'RootNode'
                        else row['metadata'].get('root_prompt')
                             if isinstance(row['metadata'], dict)
                             else None,
            axis=1
        )
        df = df.dropna(subset=['root_prompt']).copy()

        # Step 1: Declarative Generation
        # 1a) Keep only first occurrence of each root_prompt
        first_occurrences = df.drop_duplicates(subset="root_prompt").copy()

        # 1b) Convert any string‐looking‐like‐a‐list into a real Python list
        def parse_to_list(x):
            if isinstance(x, list):
                return x
            elif isinstance(x, str):
                try:
                    parsed = ast.literal_eval(x)
                    return parsed if isinstance(parsed, list) else [parsed]
                except Exception:
                    return [x]
            else:
                return [x]

        first_occurrences["possible_answers"] = first_occurrences["possible_answers"].apply(parse_to_list)

        # 1c) Flatten any nested lists one level
        def flatten_once(cell):
            if isinstance(cell, list) and len(cell) > 0 and all(isinstance(el, list) for el in cell):
                return cell[0]
            return cell

        first_occurrences["possible_answers"] = first_occurrences["possible_answers"].apply(flatten_once)

        # 1d) Explode so each row has exactly one possible_answers string
        first_occurrences = first_occurrences.explode("possible_answers").reset_index(drop=True)

        # 1e) Trim whitespace from every answer
        first_occurrences["possible_answers"] = first_occurrences["possible_answers"].astype(str).str.strip()

        # 2. Build a small HF Dataset containing exactly (question_id, root_prompt, possible_answers)
        ds = Dataset.from_pandas(
            first_occurrences[['question_id', 'root_prompt', 'possible_answers']],
            preserve_index=False
        )

        def format_inputs(example):
            prompt_text = example["root_prompt"]
            ans_text = example["possible_answers"]
            return {"input_text": f"{ans_text} \n {prompt_text}"}

        ds = ds.map(format_inputs)
        gens = declarative_generator(ds['input_text'])

        # 3. For each generated output, apply the fallback→answer logic
        declaratives = []
        for g, orig in zip(gens, ds["input_text"]):
            gen_text = g["generated_text"].strip()
            if re.search(r"\w", gen_text):
                declaratives.append(gen_text)
            else:
                answer_part = orig.split("\n", 1)[0].strip()
                declaratives.append(answer_part if answer_part else None)

        # 4. Attach those declaratives directly into first_occurrences
        first_occurrences.loc[:, "reference_ans_declarative"] = declaratives

        # 5. Merge back into df, matching only on question_id
        df = df.merge(
            first_occurrences[["question_id", "reference_ans_declarative"]],
            on=["question_id"],
            how="left"
        )
        clear_cache()

        # —────────────────────────────────────────────────────────────────—
        # Step 2: QNLI
        with get_qnli() as qnli_pipe:
            rag_sentences_col = []
            entail_probs_col = []
            entail_labels_col = []
            num_entail_col = []
            num_sent_col = []
            all_examples = []
            example_index = []

            for idx, row in df.iterrows():
                # Extract one sentence at a time from base_rag
                sentences = extract_sentences(row["base_rag"])
                rag_sentences_col.append(sentences)
                num_sent_col.append(len(sentences))

                for sent in sentences:
                    all_examples.append({"text": row["root_prompt"], "text_pair": sent})
                    example_index.append(idx)

            results = qnli_pipe(all_examples)
            entail_probs_by_row, labels_by_row, count_by_row = {}, {}, {}

            for idx in df.index:
                entail_probs_by_row[idx] = []
                labels_by_row[idx] = []
                count_by_row[idx] = 0

            for i, out_list in enumerate(results):
                idx = example_index[i]
                chosen_label = max(out_list, key=lambda d: d["score"])["label"]
                scores = {d["label"]: d["score"] for d in out_list}
                # QNLI: LABEL_0 = entailment, LABEL_1 = not entailment
                probs = [scores["LABEL_1"], scores["LABEL_0"]]
                entail_probs_by_row[idx].append(probs)
                labels_by_row[idx].append("entailment" if chosen_label == "LABEL_0" else "not entailment")
                if chosen_label == "LABEL_0":
                    count_by_row[idx] += 1

            for idx in df.index:
                entail_probs_col.append(entail_probs_by_row[idx])
                entail_labels_col.append(labels_by_row[idx])
                num_entail_col.append(count_by_row[idx])

            df.loc[:, "rag_sentences"] = rag_sentences_col
            df.loc[:, "rag_entailment_probs"] = entail_probs_col
            df.loc[:, "rag_entailment_labels"] = entail_labels_col
            df.loc[:, "num_entailments"] = num_entail_col
            df.loc[:, "num_rag_sentences"] = num_sent_col

            # Build best_rag_sentence = first sentence with “entailment” label, or None
            def _best_entailing(row):
                for sent, lab in zip(row["rag_sentences"], row["rag_entailment_labels"]):
                    if lab == "entailment":
                        return sent
                return None

            df.loc[:, "best_rag_sentence"] = df.apply(_best_entailing, axis=1)

            # Row-level QNLI: entire base_rag vs. root_prompt
            row_level_examples = []
            for _, row in df.iterrows():
                premise = row["root_prompt"]  # already a string

                # If base_rag is a list of sentences, join them; else cast to str
                if isinstance(row["base_rag"], list):
                    hypothesis = " ".join(row["base_rag"])
                else:
                    hypothesis = str(row["base_rag"])

                row_level_examples.append({
                    "text": premise,
                    "text_pair": hypothesis
                })

            row_level_results = qnli_pipe(row_level_examples)

            base_labels = [
                1 if max(r, key=lambda d: d["score"])["label"] == "LABEL_0" else 0
                for r in row_level_results
            ]
            base_probs = [[d["score"] for d in r] for r in row_level_results]

            df.loc[:, "base_rag_label"] = base_labels
            df.loc[:, "base_rag_probs"] = base_probs

        clear_cache()

        # —────────────────────────────────────────────────────────────────— 
        # Step 3: MNLI (using BART‐MNLI)
        df = df.copy()
        with get_mnli() as mnli_pipe:
            # Build each input as "best_rag_sentence \n root_prompt"
            rag_inputs = [
                f"{sent.strip()} \n {prompt}"
                if isinstance(sent, str) and sent.strip()
                else ""
                for sent, prompt in zip(df["best_rag_sentence"], df["root_prompt"])
            ]

            # 3a) Call the generator on those combined strings
            rag_gens = declarative_generator(rag_inputs)

            # 3b) Fallback: if generated_text is blank, return only the sentence portion
            rag_declaratives = []
            for gen_dict, orig_input in zip(rag_gens, rag_inputs):
                generated = gen_dict["generated_text"].strip()
                if re.search(r"\w", generated):
                    rag_declaratives.append(generated)
                else:
                    sentence_part = orig_input.split("\n", 1)[0].strip()
                    rag_declaratives.append(sentence_part if re.search(r"\w", sentence_part) else None)

            df.loc[:, "rag_declarative"] = rag_declaratives

            # 3c) Build MNLI examples ("text" = premise, "text_pair" = hypothesis)
            mnli_examples = []
            for _, row in df.iterrows():
                # 1) Get premise; if it's a list, join it, otherwise take the string or ""
                raw_premise = row["rag_declarative"]
                if isinstance(raw_premise, list):
                    premise = " ".join(raw_premise)
                else:
                    premise = raw_premise or ""

                # 2) Get hypothesis; if it's a list, join it, otherwise take the string or ""
                raw_hypothesis = row["reference_ans_declarative"]
                if isinstance(raw_hypothesis, list):
                    hypothesis = " ".join(raw_hypothesis)
                else:
                    hypothesis = raw_hypothesis or ""

                mnli_examples.append({
                    "text": premise,
                    "text_pair": hypothesis
                })

            # 3d) Run MNLI
            mnli_results = mnli_pipe(mnli_examples)
            best_labels = []
            all_probs   = []
            for out in mnli_results:
                best = max(out, key=lambda d: d["score"])
                best_labels.append(best["label"])
                scores_dict = {d["label"]: d["score"] for d in out}
                all_probs.append([
                    scores_dict.get("entailment",   0.0),
                    scores_dict.get("contradiction", 0.0),
                    scores_dict.get("neutral",       0.0)
                ])

            df.loc[:, "mnli_label_rag_to_ref"] = best_labels
            df.loc[:, "mnli_probs_rag_to_ref"] = all_probs

        clear_cache()
        # 1) Convert mnli_label_rag_to_ref: "entailment" → 1, everything else → 0
        df['mnli_label_rag_to_ref'] = df['mnli_label_rag_to_ref'].apply(lambda x: 1 if x == 'entailment' else 0)

        # 2) Drop the 'reference_ans_declarative' column
        df = df.drop(columns=['reference_ans_declarative'])

        # 3) Get the list of remaining columns
        df_columns = list(df.columns)
        print("All columns before grouping:", df_columns)

        # 4) Group by question_id:
        #
        #    - For mnli_label_rag_to_ref: take the maximum within each question_id group.
        #    - For every other column: collect unique values as a numpy array.
        #
        aggregation = {}
        for col in df_columns:
            if col == 'question_id':
                # question_id is the grouping key; no aggregation needed.
                continue
            elif col == 'mnli_label_rag_to_ref':
                aggregation[col] = 'max'
            else:
                aggregation[col] = 'first'

        # Perform the groupby‐aggregation
        df_grouped = df.groupby('question_id').agg(aggregation).reset_index()

        # Now each column (other than mnli_label_rag_to_ref) is an array of unique values
        # and mnli_label_rag_to_ref is either 0 or 1 per question_id.
        print("Columns after grouping:", list(df_grouped.columns))

        # If you want df_columns to refer specifically to the post‐grouping columns:
        df_columns = list(df_grouped.columns)
        print("All columns after grouping:", df_columns)

        # For convenience, you can flatten any single‐element arrays back into scalars. 
        # For example, if you know “root_prompt” was identical for every row of a question, you can do:
        for col in df_columns:
            if col == 'question_id' or col == 'mnli_label_rag_to_ref':
                continue
            # If a column’s values are always length‐1 arrays, extract that single value
            df_grouped[col] = df_grouped[col].apply(lambda arr: arr[0] if isinstance(arr, (list, pd.Series, np.ndarray)) and len(arr) == 1 else arr)

        updated_dfs.append(df_grouped)

    clear_cache()
    return updated_dfs


In [98]:
declarative_generator = pipeline(
    "text2text-generation",
    model='khhuang/zerofec-qa2claim-t5-base',
    tokenizer='khhuang/zerofec-qa2claim-t5-base',
    batch_size=512,
    return_tensors=False
)

Device set to use cuda:0


In [106]:
df = read_dataset('POPQA', 'paraprefix')
df = df.iloc[:500]
df_list = [df]

updated_dfs = evaluate_pipeline(
    df_list=df_list,
    declarative_generator=declarative_generator,
)

Map:   0%|          | 0/109 [00:00<?, ? examples/s]

/vol/bitbucket/lst20/lex-eval/lib/python3.12/site-packages/torch/__init__.py:1113: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
Some weights of the model checkpoint at textattack/roberta-base-QNLI were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
/vol/bitbucket/lst20/lex-eval/lib/python3.12/site-

All columns before grouping: ['Unnamed: 0', 'gen_modelId', 'question_id', 'layer', 'id', 'type', 'para_type', 'prompt', 'rag_closest_match', 'rag_entities', 'possible_answers', 'metadata', 'wiki_title', 'rag_found_match', 'base_found_match', 'base_f1_1', 'base_f1_L', 'rag_f1_1', 'rag_f1_L', 'base_rec_1', 'base_rec_L', 'rag_rec_1', 'rag_rec_L', 'base_prec_1', 'base_prec_L', 'rag_prec_1', 'rag_prec_L', 'root_similarity_score', 'complexity_score', 'fk_score', 'dc_score', 'similarity', 'base', 'base_rag', 'root_prompt', 'rag_sentences', 'rag_entailment_probs', 'rag_entailment_labels', 'num_entailments', 'num_rag_sentences', 'best_rag_sentence', 'base_rag_label', 'base_rag_probs', 'rag_declarative', 'mnli_label_rag_to_ref', 'mnli_probs_rag_to_ref']
Columns after grouping: ['question_id', 'Unnamed: 0', 'gen_modelId', 'layer', 'id', 'type', 'para_type', 'prompt', 'rag_closest_match', 'rag_entities', 'possible_answers', 'metadata', 'wiki_title', 'rag_found_match', 'base_found_match', 'base_f1_

/vol/bitbucket/lst20/lex-eval/lib/python3.12/site-packages/torch/__init__.py:1113: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


In [107]:
updated_dfs[0][['mnli_label_rag_to_ref', 'possible_answers', 'base_rag', 'rag_declarative', 'possible_answers']]

,mnli_label_rag_to_ref,possible_answers,base_rag,rag_declarative,possible_answers
0,0,"['Des Moines', 'Des Moines, Iowa']",Iowa City.,Iowa City is the capital of Iowa.,"['Des Moines', 'Des Moines, Iowa']"
1,1,"['Rick Rubin', 'Frederick Jay Rubin', 'DJ Doub...",Rick Rubin.,Rick Rubin was the producer of 13.,"['Rick Rubin', 'Frederick Jay Rubin', 'DJ Doub..."
2,0,['Joyce Summers'],Kristine Sutherland.,Kristine Sutherland is the mother of Dawn Summ...,['Joyce Summers']
3,1,['LeVar Burton'],LeVar Burton.,LeVar Burton was the director of Second Chances.,['LeVar Burton']
4,0,['Martin Ritt'],Phil Sussman,Phil Sussman was the producer of The Front.,['Martin Ritt']
5,1,"['Terence Winter', 'Terence Patrick Winter']",Terence Winter.,Terence Winter was the screenwriter for The We...,"['Terence Winter', 'Terence Patrick Winter']"
6,1,['Lysimachus'],Lysimachus of Telmessos.,Lysimachus of Telmessos is the father of Ptole...,['Lysimachus']
7,1,"['Paramount Pictures', 'Paramount Pictures Cor...",Alfred Hitchcock.,Alfred Hitchcock was the producer of Vertigo.,"['Paramount Pictures', 'Paramount Pictures Cor..."
8,1,"['India', 'Bharat', 'Hindustan', 'Bharatvarsh'...",India.,Seemant Institute of Technology is in India.,"['India', 'Bharat', 'Hindustan', 'Bharatvarsh'..."
9,1,"['surgeon', 'surgeons']",Scottish anatomist and surgeon. Farmer. Politi...,John Bell's occupation is Scottish anatomist a...,"['surgeon', 'surgeons']"
